### SCPT to VS
Computation and plotting follow `SCPT_Computation.ipynb` (Zhang et al. global & site-specific, Robertson (2012), LBR).

In [1]:
# Import packages for use:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
import ngl_tools.smt as smt
from layer_based import LBR

plt.rcParams['font.family'] = 'Times New Roman'

pd.set_option('display.max_columns', None)

In [2]:
path = Path("..").joinpath(
    "src_files"
)
cpt_data = pd.read_csv(path.joinpath("cpt_data_table.csv"))
cpt_meta = pd.read_csv(path.joinpath("cpt_meta_table.csv"))
traveltime_meta = pd.read_csv(path.joinpath("traveltime_meta_table.csv"))
traveltime_data = pd.read_csv(path.joinpath("traveltime_data_table.csv"))
test = pd.read_csv(path.joinpath("test_table.csv"))
geology = pd.read_csv(path.joinpath("surficial_geology.csv"))

cpt_meta = cpt_meta.rename(columns={'id': 'cpt_id'})
traveltime_meta = traveltime_meta.rename(columns={'id': 'traveltime_id'})
traveltime_meta_joint = traveltime_meta.merge(cpt_meta[['cpt_id', 'test_id']], on='test_id', how='left')

try:
    REVIEW_DF = pd.read_csv("traveltime_review.csv")
except FileNotFoundError:
    REVIEW_DF = traveltime_meta[['traveltime_id', 'test_id']].rename(columns={'traveltime_id': 'travelTimeMeta_ID'})
    REVIEW_DF['status'] = 'Pending'
    REVIEW_DF['VsZ_Direct_interpretation'] = np.nan
    REVIEW_DF['VsZ_Global'] = np.nan
    REVIEW_DF['VsZ_Site_Specific'] = np.nan
    REVIEW_DF['VsZ_Robertson'] = np.nan
    REVIEW_DF['VsZp_Direct_interpretation'] = np.nan
    REVIEW_DF['comment'] = np.nan
    REVIEW_DF.to_csv("traveltime_review.csv", index=False, header=True)

try:
    VS_SLOPEBREAK_DF = pd.read_csv("slopebreak_vs.csv")
except FileNotFoundError:
    VS_SLOPEBREAK_DF = pd.DataFrame(columns=['travelTimeMeta_ID', 'Velocity', 'Top_depth', 'Bottom_depth'])
    VS_SLOPEBREAK_DF.to_csv("slopebreak_vs.csv", index=False, header=True)

In [3]:
def query_data(id):
    
    tt_data = traveltime_data[traveltime_data['travel_time_metadata_id'] == id].reset_index(drop=True)
    tt_depth = tt_data['pick_depth'].values
    traveltime = tt_data['pick_time'].values

    valid_mask = ~np.isnan(tt_depth) & ~np.isnan(traveltime)
    tt_depth = tt_depth[valid_mask]
    traveltime = traveltime[valid_mask]

    cpt_id = traveltime_meta_joint[traveltime_meta_joint['traveltime_id'] == id]['cpt_id'].values[0]
    data = cpt_data[cpt_data['cpt_metadata_id'] == cpt_id].reset_index(drop=True)
    cpt_depth = data['cpt_depth'].values
    qc = data['tip_resistance'].values
    fs = data['sleeve_friction'].values

    valid_mask = ~np.isnan(cpt_depth) & ~np.isnan(qc) & ~np.isnan(fs)
    cpt_depth = cpt_depth[valid_mask]
    qc = qc[valid_mask]
    fs = fs[valid_mask]

    return(tt_depth, traveltime, cpt_depth, qc, fs)


def query_DGWT(travelTimeMeta_ID):
    # Update on 2026-09-28: groundwater table query from Ma et al. 2025 and USGS topo
    DGWT_DATA = pd.read_csv(path.joinpath("gwt_master_table.csv"))
    index = DGWT_DATA['id'] == travelTimeMeta_ID
    dgwt = DGWT_DATA['dgwt_m'][index].values
    
    return dgwt

def gamma_t (depth, fs, qt):

    # Replace invalid values in CPT_fs and CPT_qt with NaN
    fs1 = fs.copy()
    qt1 = qt.copy()
    fs1[fs1 <= 0] = np.nan
    qt1[qt1 <= 0] = np.nan
    valid_mask = ~np.isnan(fs1) & ~np.isnan(qt1)

    gamma = np.full_like(depth, np.nan)  # Initialize with NaN

    gamma[valid_mask] = 9.81 * (0.27 * np.log10(fs1[valid_mask]/qt1[valid_mask] * 100) + 0.36 * np.log10(qt1[valid_mask] / 101.3) + 1.236)

     # Interpolate NaN values
    gamma = np.interp(depth, depth[valid_mask], gamma[valid_mask])

    return gamma

def interleave(v1, v2):
    vout = np.empty(2 * len(v1), dtype=float)
    vout[0::2] = v1
    vout[1::2] = v2
    return vout 

# Function to get sigmavp, Qtn and Ic

def Preprocessing_TT (travelTimeMeta_ID, source = 1.0):
    TT_depth, Meas_TT, CPT_depth, CPT_qt, CPT_fs = query_data(travelTimeMeta_ID)

    # Correction of Source Offset Distance
    TT = TT_depth / np.sqrt (TT_depth**2 + source**2) * Meas_TT

    slowness_TT = np.zeros(len(TT)-1)
    slowness_TT_depth = np.zeros(len(TT)-1)
    for i in range(len(TT)-1):
        slowness_TT[i] = (TT[i+1] - TT[i]) / (TT_depth[i+1] - TT_depth[i]) /1000
        slowness_TT_depth[i] = (TT_depth[i+1] + TT_depth[i]) / 2
  
    ztop_TT = TT_depth[:-1]
    zbot_TT = TT_depth[1:]

    halfdepth = slowness_TT_depth

    DTT = np.diff(TT)

    return TT, TT_depth, halfdepth, DTT, slowness_TT, slowness_TT_depth, ztop_TT, zbot_TT

def Vs_interval (traveltimeMeta_ID):
    TT, TT_depth, halfdepth, DTT, slowness_TT, slowness_TT_depth, ztop_TT, zbot_TT = Preprocessing_TT(traveltimeMeta_ID)
    Vsi = (zbot_TT - ztop_TT)/(DTT/1000)
    return Vsi

def Robertson (Qtn, Ic, fz):
    Vs = np.exp(1.93 + 0.5 * np.log(Qtn) + 0.25 * np.log(fz) + 0.63 * Ic)
    return Vs

def get_CPT (travelTimeMeta_ID):
    TT_depth, TT, CPT_depth, CPT_qt, CPT_fs = query_data(travelTimeMeta_ID)

    dGWT = query_DGWT(travelTimeMeta_ID)
    CPT_qt = CPT_qt * 1000
    CPT_fs = CPT_fs * 1000 

    pa = 101.3 #KPa,
    gamma = gamma_t(CPT_depth, CPT_fs, CPT_qt)

    CPT_fs[CPT_fs < 0] = 0.01
    CPT_qt[CPT_qt < 0] = 0.01

    # integrate gamma over CPT_depth to get sigmav, then compute sigmavp by subtracting the pore pressure u
    # updated 2026-02-24
    depth_interval = np.diff(CPT_depth, prepend=0)
    sigmav = np.cumsum(gamma * depth_interval)
           
    u = (CPT_depth - dGWT) * 9.81
    u[u<0] = 0
    sigmavp = sigmav - u

    fz = sigmavp/pa

    # Generate the layered Ic
    qt_inv, fs_inv, Ic_inv = smt.cpt_inverse_filter(CPT_qt, CPT_depth, fs=CPT_fs, sigmav=sigmav, sigmavp=sigmavp)
    Ic_inv, Qtn_inv, Fr_inv = smt.get_Ic_Qtn_Fr(qt_inv, fs_inv, sigmav, sigmavp)

    Qtncs = Qtn_inv.copy()
    kc = np.full(Qtn_inv.shape, 1.0) #array of ones for kc

    Qtncs = Qtn_inv * kc

    return fz, Ic_inv, Qtncs, CPT_depth, Qtn_inv, Ic_inv

def get_CPT_Robertson(travelTimeMeta_ID):
    TT_depth, TT, CPT_depth, CPT_qt, CPT_fs = query_data(travelTimeMeta_ID)

    dGWT = query_DGWT(travelTimeMeta_ID)
    CPT_qt = CPT_qt * 1000
    CPT_fs = CPT_fs * 1000 

    pa = 101.3 #KPa,
    gamma = gamma_t(CPT_depth, CPT_fs, CPT_qt)

    CPT_fs[CPT_fs < 0] = 0.01
    CPT_qt[CPT_qt < 0] = 0.01

    # integrate gamma over CPT_depth to get sigmav, then compute sigmavp by subtracting the pore pressure u
    # updated 2026-02-24
    depth_interval = np.diff(CPT_depth, prepend=0)
    sigmav = np.cumsum(gamma * depth_interval)
           
    u = (CPT_depth - dGWT) * 9.81
    u[u<0] = 0
    sigmavp = sigmav - u

    fz = sigmavp/pa

    # Generate the layered Ic
    qt_inv, fs_inv, Ic_inv = smt.cpt_inverse_filter(CPT_qt, CPT_depth, fs=CPT_fs, sigmav=sigmav, sigmavp=sigmavp)
    Ic, Qtn, Fr = smt.get_Ic_Qtn_Fr(CPT_qt, CPT_fs, sigmav, sigmavp)
    Ic_inv, Qtn_inv, Fr_inv = smt.get_Ic_Qtn_Fr(qt_inv, fs_inv, sigmav, sigmavp)

    return fz, Ic, Qtn, CPT_depth

# def TT_average (travelTimeMeta_ID, Kc_c1, Kc_c2, source = 1.0):
def TT_average (travelTimeMeta_ID, source = 1.0):
    ztop_TT, zbot_TT = Preprocessing_TT(travelTimeMeta_ID, source = source)[-2:]
    fz, Ic_inv, Qtn, CPT_depth, Qtn_inv, Ic_inv = get_CPT(travelTimeMeta_ID)
    
    Qtn_TT = np.zeros(len(ztop_TT))
    Ic_TT = np.zeros(len(ztop_TT))
    fz_TT = np.zeros(len(ztop_TT))
    
    for i in range(len(ztop_TT)):
        Qtn_TT[i] =np.mean(Qtn[(CPT_depth >= ztop_TT[i]) & (CPT_depth < zbot_TT[i])])
        Ic_TT[i] = np.mean(Ic_inv[(CPT_depth >= ztop_TT[i]) & (CPT_depth < zbot_TT[i])])
        fz_TT[i] = np.mean(fz[(CPT_depth >= ztop_TT[i]) & (CPT_depth < zbot_TT[i])])

    return Qtn_TT, Ic_TT, fz_TT, ztop_TT, zbot_TT

def CPT_data(travelTimeMeta_ID): 
    fz, Ic, Qtn, CPT_depth, Qtn_inv, Ic_inv = get_CPT(travelTimeMeta_ID)

    valid_indices = (fz > 0) & (Qtn > 0) & (Ic >= 1)

    Qtn = np.asarray(Qtn[valid_indices])
    Ic = np.asarray(Ic[valid_indices])
    fz = np.asarray(fz[valid_indices])
    CPT_depth = np.asarray(CPT_depth[valid_indices])
    Qtn_inv = np.asarray(Qtn_inv[valid_indices])
    Ic_inv = np.asarray(Ic_inv[valid_indices])

    tt_id = np.full(len(CPT_depth), travelTimeMeta_ID)

    return tt_id, CPT_depth, fz, Ic, Qtn, Qtn_inv, Ic_inv

def interleave(v1, v2):
    vout = np.empty(2 * len(v1), dtype=float)
    vout[0::2] = v1
    vout[1::2] = v2
    return vout 

def site_specific(x, CPT_depth, Qtn, sigmavp, Ic, ztop, zbot, DTT): 

    in_layer = (CPT_depth[:, None] >= ztop[None, :]) & (CPT_depth[:, None] <= zbot[None, :])
    pair_cpt, pair_layer = np.nonzero(in_layer)

    n_layers = len(DTT)
    N_layer = np.bincount(pair_layer, minlength=n_layers)
    has_pts = (N_layer > 0) & (DTT > 0)

    Qtn = Qtn[pair_cpt]
    sigmavp = sigmavp[pair_cpt]
    Ic = Ic[pair_cpt]

    kc = 1 + 5 / (1 + np.exp(-5.482742 * Ic + 12.320768))
    m = 0.1 + 0.4 / (1 + np.exp(-1.278293 * Ic + 3.385425))

    velocity = 0.224117 * np.log(kc * Qtn) + m * np.log(sigmavp) + x
    cpt_slowness = 1.0 / np.exp(velocity)
    sum_slowness = np.bincount(pair_layer, weights=cpt_slowness, minlength=n_layers)

    dz = (zbot - ztop)[has_pts] / N_layer[has_pts]  # each CPT point represents an equal share of its layer
    tt_hat = dz * sum_slowness[has_pts]
    ln_tt = np.log(DTT[has_pts] / 1000)
    residual =  ln_tt - np.log(tt_hat)

    return residual

def calc_tt_hat(x, CPT_depth, Qtn, sigmavp, Ic, ztop, zbot, DTT): 

    in_layer = (CPT_depth[:, None] >= ztop[None, :]) & (CPT_depth[:, None] <= zbot[None, :])
    pair_cpt, pair_layer = np.nonzero(in_layer)

    n_layers = len(DTT)
    N_layer = np.bincount(pair_layer, minlength=n_layers)
    has_pts = (N_layer > 0) & (DTT > 0)

    Qtn = Qtn[pair_cpt]
    sigmavp = sigmavp[pair_cpt]
    Ic = Ic[pair_cpt]

    kc = 1 + 5 / (1 + np.exp(-5.482742 * Ic + 12.320768))
    m = 0.1 + 0.4 / (1 + np.exp(-1.278293 * Ic + 3.385425))

    velocity = 0.224117 * np.log(kc * Qtn) + m * np.log(sigmavp) + x
    cpt_slowness = 1.0 / np.exp(velocity)
    sum_slowness = np.bincount(pair_layer, weights=cpt_slowness, minlength=n_layers)

    dz = (zbot - ztop)[has_pts] / N_layer[has_pts]  # each CPT point represents an equal share of its layer
    tt_hat = dz * sum_slowness[has_pts] * 1000
    halfdepth = 0.5 * (ztop[has_pts] + zbot[has_pts])

    return tt_hat, halfdepth

In [4]:
# VsZ (time-averaged velocity from the surface to the bottom of each profile): VsZ = Z / sum(h_i / Vs_i)
def VsZ_CPT(depth, velocity):
    # Each CPT point represents the interval above it (first point extends to the surface)
    valid = np.isfinite(depth) & np.isfinite(velocity) & (velocity > 0)
    depth = depth[valid]
    velocity = velocity[valid]
    dz = np.diff(depth, prepend=0)
    Z = depth[-1]
    return Z / np.sum(dz / velocity), Z

def VsZ_layered(zbot, velocity):
    # Layers are contiguous from the surface (first ztop = 0.01 is treated as 0)
    zbot = np.asarray(zbot, dtype=float)
    dz = np.diff(zbot, prepend=0)
    Z = zbot[-1]
    return Z / np.sum(dz / np.asarray(velocity, dtype=float)), Z

a = [4.402626, 4.255830, 4.195240, 4.195468, 4.238807, 4.316504, 4.350243, 4.411610, 4.383580, 4.515429, 4.420910]

def compute_profiles(ID, slope_break_method = 0, breakpoints = None):

    tt_id, CPT_depth, sigmavp, Ic, Qtn, Qtn_inv, Ic_inv = CPT_data(ID)

    kc = 1 + 5 / (1 + np.exp(-5.482742 * Ic + 12.320768))
    m = 0.1 + 0.4 / (1 + np.exp(-1.278293 * Ic + 3.385425))

    geo = geology['group_id'][geology['travelTimeMeta_ID'] == ID].iloc[0]
    a_geo = a[geo-1]

    # Compute global velocity using Zhang et al. method
    global_velocity = np.exp(0.224117 * np.log(kc * Qtn) + m * np.log(sigmavp) + a_geo)

    # Compute Robertson Vs
    sigmavp_rob, Ic_rob, Qtn_rob, CPT_depth_rob = get_CPT_Robertson(ID)
    Robertson_Vs = np.exp(1.93 + 0.5 * np.log(Qtn_rob) + 0.25 * np.log(sigmavp_rob) + 0.63 * Ic_rob)

    # Compute Vsi
    TT, TT_depth, halfdepth, DTT, slowness_TT, slowness_TT_depth, ztop_TT, zbot_TT = Preprocessing_TT(ID)
    Vsi = (zbot_TT - ztop_TT) / DTT * 1000

    # Compute site-specific velocity with Zhang et al. method
    a_0 = 4.0
    result = least_squares(site_specific, x0=a_0, args=(CPT_depth, Qtn, sigmavp, Ic, ztop_TT, zbot_TT, DTT))
    a_site_specific = result.x[0]
    site_specific_velocity = np.exp(0.224117 * np.log(kc * Qtn) + m * np.log(sigmavp) + a_site_specific)
    tt_hat_ss, halfdepth_ss = calc_tt_hat(a_site_specific, CPT_depth, Qtn, sigmavp, Ic, ztop_TT, zbot_TT, DTT)
    tt_hat_global, halfdepth_global = calc_tt_hat(a_geo, CPT_depth, Qtn, sigmavp, Ic, ztop_TT, zbot_TT, DTT)

    # Compute LBR velocity (automatic, or manual with the selected breakpoints)
    tt_depth, traveltime, cpt_depth, qc, fs = query_data(ID)
    LBR_velocity, tt_corrected, results, LBR_ztop, LBR_zbot = LBR(tt_depth, traveltime, slope_break_method = slope_break_method, breakpoints = breakpoints)

    return dict(ID=ID, geo=geo, a_geo=a_geo, a_site_specific=a_site_specific,
                CPT_depth=CPT_depth, sigmavp=sigmavp, Ic=Ic, Qtn=Qtn, kc=kc,
                global_velocity=global_velocity, site_specific_velocity=site_specific_velocity,
                Robertson_Vs=Robertson_Vs, CPT_depth_rob=CPT_depth_rob,
                TT=TT, TT_depth=TT_depth, halfdepth=halfdepth, DTT=DTT, ztop_TT=ztop_TT, zbot_TT=zbot_TT, Vsi=Vsi,
                tt_hat_ss=tt_hat_ss, halfdepth_ss=halfdepth_ss, tt_hat_global=tt_hat_global, halfdepth_global=halfdepth_global,
                LBR_velocity=LBR_velocity, results=results, LBR_ztop=LBR_ztop, LBR_zbot=LBR_zbot)

def plot_profiles(p):
    CPT_depth, Qtn, kc, sigmavp, Ic = p['CPT_depth'], p['Qtn'], p['kc'], p['sigmavp'], p['Ic']
    TT, TT_depth, halfdepth, DTT = p['TT'], p['TT_depth'], p['halfdepth'], p['DTT']
    results = p['results']

    fig, axs = plt.subplots(1, 6, figsize=(12, 5), dpi = 300)

    axs[0].plot(Qtn, CPT_depth, label='Qtn', color = 'grey', linewidth = 1.2)
    axs[0].plot(Qtn * kc, CPT_depth, label='Qtncs', color='tab:blue', linewidth=1.2)
    axs[0].set_xlim(0, 1000)
    axs[0].set_xticks(np.arange(0, 1001, 200))
    axs[0].grid(True, which="both", ls="--")
    axs[0].set_xlabel('Qtn / Qtncs')
    axs[0].set_ylabel('Depth (m)')
    axs[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

    axs[1].plot((0, sigmavp[0]), (0, CPT_depth[0]), color='tab:blue', linestyle='dashed', linewidth = 1.2)
    axs[1].plot(sigmavp, CPT_depth, label='$\\sigma_v^\\prime$_CPT', color='tab:blue', linewidth = 1.2)
    axs[1].set_xlabel('$\\sigma_v^\\prime$ (atm)')
    axs[1].set_xticks(np.arange(0, 11, 2))
    axs[1].set_xlim(0, 10)
    axs[1].grid(True, which="both", ls="--")
    axs[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

    axs[2].plot(Ic, CPT_depth, label='$I_c$', color = 'tab:blue', linewidth = 1.2)
    axs[2].set_xlim(1, 4)
    axs[2].grid(True, which="both", ls="--")
    axs[2].set_xlabel('Ic')
    axs[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

    axs[3].scatter(DTT, halfdepth, label='Measured $\\Delta$tt', marker='o', edgecolor='black', color = 'white', s=10)
    axs[3].scatter(p['tt_hat_global'], p['halfdepth_global'], label='Predicted $\\Delta$tt', marker='x', color='tab:blue', s=10)
    axs[3].scatter(p['tt_hat_ss'], p['halfdepth_ss'], label='Predicted $\\Delta$tt (ss)', marker='x', color='tab:orange', s=10)
    axs[3].set_xlabel('$\\Delta$tt (ms)')
    axs[3].set_xlim(0, 10)
    axs[3].set_xticks(np.arange(0, 11, 2))
    axs[3].grid(True, which="both", ls="--")
    axs[3].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

    axs[4].scatter(TT, TT_depth, label='Travel Time Data', color='black', s=10)
    colors = plt.colormaps['tab10']

    # Plot each segment with a different color
    for i in range(len(results['breaks']) - 1):
        # Define segment depth range
        z_start = results['breaks'][i]
        z_end = results['breaks'][i + 1]

        # Get depths in this segment (new_depths ensures breaks are included)
        segment_mask = (results['new_depths'] >= z_start) & (results['new_depths'] <= z_end)
        segment_depths = results['new_depths'][segment_mask]
        segment_tt = results['fitted_values_for_plot'][segment_mask]

        # Plot segment
        axs[4].plot(segment_tt, segment_depths, color=colors(i))

    axs[4].set_xlabel('Travel Time (ms)')
    axs[4].set_xlim(0, np.max(TT) * 1.1)
    if np.max(TT) > 150:
        axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 50))
    elif np.max(TT) > 90:
        axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 30))
    elif np.max(TT) > 50:
        axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 20))
    else:
        axs[4].set_xticks(np.arange(0, np.max(TT) * 1.1, 10))
    axs[4].grid(True, which="both", ls="--")
    axs[4].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)

    axs[5].plot(p['global_velocity'], CPT_depth, label='Zhang et al.', color='tab:blue', linewidth = 1.2)
    axs[5].plot(p['site_specific_velocity'], CPT_depth, label='Zhang et al. (ss)', color='tab:orange', linewidth = 1.2)
    axs[5].plot(p['Robertson_Vs'], p['CPT_depth_rob'], label='Robertson (2012)', color='grey', linewidth = 1.2)
    axs[5].plot(interleave(p['LBR_velocity'], p['LBR_velocity']), interleave(p['LBR_ztop'], p['LBR_zbot']), label='LBR', color='black', linewidth = 1.2)
    axs[5].set_xlim(0, 1000)
    axs[5].set_xticks(np.arange(0, 1001, 200))
    axs[5].set_xlabel('Velocity (m/s)')
    axs[5].legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8)
    axs[5].grid(True, which="both", ls="--")

    # Share common y-axis limits across all subplots
    y_max = np.nanmax([np.nanmax(CPT_depth), np.nanmax(p['zbot_TT']), np.nanmax(TT_depth)])
    for ax in axs:
        ax.set_ylim(y_max + 1, 0)  # inverted depth axis, same range for all

    # Panel labels (a)-(f) in the top right corner of each subplot
    for ax, panel in zip(axs, 'abcdef'):
        ax.text(0.95, 0.97, f'({panel})', transform=ax.transAxes, ha='right', va='top', fontsize=10,
                bbox=dict(facecolor='white', edgecolor='none', pad=1.5))

    plt.subplots_adjust(bottom=0.28, wspace=0.35)
    plt.tight_layout()
    plt.show()

def compute_VsZ(p):
    VsZ_results = {
        'Zhang et al.': VsZ_CPT(p['CPT_depth'], p['global_velocity']),
        'Zhang et al. (ss)': VsZ_CPT(p['CPT_depth'], p['site_specific_velocity']),
        'Robertson (2012)': VsZ_CPT(p['CPT_depth_rob'], p['Robertson_Vs']),
        'LBR': VsZ_layered(p['LBR_zbot'], p['LBR_velocity']),
    }

    # VsZp: LBR time-averaged velocity from the first travel time depth to the bottom of the profile
    top_depth_lim = np.asarray(p['LBR_ztop'], dtype=float)
    top_depth_lim[0] = np.min(p['TT_depth'])
    VsZp = (np.max(p['TT_depth']) - np.min(p['TT_depth'])) / np.sum((np.asarray(p['LBR_zbot']) - top_depth_lim) / np.asarray(p['LBR_velocity']))

    return VsZ_results, VsZp

### GUI

In [5]:
import ipywidgets as widgets
from ipywidgets import HTML, Layout, HBox, VBox, Dropdown, FloatText, Button, Textarea
from IPython.display import display

%matplotlib widget

# Setup
style = {'description_width': 'initial'}
starting_index = REVIEW_DF[REVIEW_DF['status'] == 'Pending'].index[0]
travelTimeMeta_ID = REVIEW_DF.iloc[starting_index]['travelTimeMeta_ID']

velocity_updated = []
top_depth_updated = []
bottom_depth_updated = []
current_fig = None
current_p = None

TRAVELTIME = Dropdown(options=REVIEW_DF['travelTimeMeta_ID'].unique(), value=travelTimeMeta_ID, description='Travel Time ID', style=style)
VsZ_input = FloatText(value=REVIEW_DF.iloc[starting_index]['VsZ_Direct_interpretation'], description='VsZ Direct Interpretation (LBR)', style=style)
# VsZp_input = FloatText(value=REVIEW_DF.iloc[starting_index]['VsZp_Direct_interpretation'], description='VsZp Direct Interpretation (LBR)', style=style)
VsZ_model_input = FloatText(value=REVIEW_DF.iloc[starting_index]['VsZ_Global'], description='VsZ Zhang et al.', style=style)
VsZ_ss_input = FloatText(value=REVIEW_DF.iloc[starting_index]['VsZ_Site_Specific'], description='VsZ Zhang et al. (ss)', style=style)
VsZ_rob_input = FloatText(value=REVIEW_DF.iloc[starting_index]['VsZ_Robertson'], description='VsZ Robertson (2012)', style=style)

button_layout = widgets.Layout(width='300px', height='40px')
ACCEPT = Button(description='Accept', layout = button_layout)
REVISE_FULL_HAND = Button(description='Full Manual Revision', layout = button_layout)
REJECT = Button(description='Reject', layout = button_layout)
SAVE_FIG = Button(description='Save Profile as Image', layout = button_layout)
COMMENT = Textarea(description='Review Comments:', value=None, style=style, layout=Layout(width='30%'))

# Dynamic checkbox container (empty to start)
checkbox_box = widgets.VBox(layout=Layout(flex_flow='row wrap', width='100%'))

box_layout_2 = widgets.Layout(width='100%', border='solid 2px', padding='20px')
SEARCH = VBox([
    HTML('<center><font size="+1.5"><b>SCPT Travel Time Interpretation Automated Tool</b>'),
    HBox([TRAVELTIME], layout=Layout(width='100%')),
    HBox([VsZ_input, VsZ_model_input, VsZ_ss_input, VsZ_rob_input], layout=Layout(width='100%')),
    HBox([ACCEPT, REVISE_FULL_HAND, REJECT, SAVE_FIG, COMMENT], layout=Layout(width='100%')),
    checkbox_box  # add the checkboxes below the buttons
], layout=box_layout_2)

out = widgets.Output()

def Run_and_Plot(ID, slope_break_method = 0, breakpoints = None):

    global velocity_updated
    global top_depth_updated
    global bottom_depth_updated
    global current_fig
    global current_p

    plt.close('all')
    p = compute_profiles(ID, slope_break_method = slope_break_method, breakpoints = breakpoints)

    print(f"Selected ID: {ID}  |  geology group_id = {p['geo']}  |  a_geo = {p['a_geo']:.4f}  |  a_ss = {p['a_site_specific']:.4f}")
    plot_profiles(p)
    current_fig = plt.gcf()
    current_p = p

    VsZ_results, VsZp = compute_VsZ(p)
    VsZ_table = pd.DataFrame([{'Method': k, 'Z (m)': Z, 'VsZ (m/s)': v} for k, (v, Z) in VsZ_results.items()])
    # display(VsZ_table)

    VsZ_input.value = VsZ_results['LBR'][0]
    # VsZp_input.value = VsZp
    VsZ_model_input.value = VsZ_results['Zhang et al.'][0]
    VsZ_ss_input.value = VsZ_results['Zhang et al. (ss)'][0]
    VsZ_rob_input.value = VsZ_results['Robertson (2012)'][0]

    velocity_updated = p['LBR_velocity']
    top_depth_updated = p['LBR_ztop']
    bottom_depth_updated = p['LBR_zbot']

    return p

# --- Reactive SCPT update ---
def Update_SCPT(_):
    out.clear_output()
    with out:
        p = Run_and_Plot(TRAVELTIME.value, slope_break_method = 0)

        # Update the checkboxes dynamically
        results = p['results']
        breaks = sorted(set(results.get('candidate_breakpoints', [])))
        used_breaks = [str(round(b, 3)) for b in results.get('breaks', [])]
        checkbox_box.children = [
            widgets.Checkbox(description=str(round(b, 3)), indent=False, value=(str(round(b, 3)) in used_breaks))
            for b in breaks
        ]

def Full_Manual(_):
    out.clear_output()
    with out:
        # Get selected breakpoints
        selected_breakpoints = [float(checkbox.description) for checkbox in checkbox_box.children if checkbox.value]
        if not selected_breakpoints:
            print("No breakpoints selected.")
            return
        # Run full manual regression
        try:
            Run_and_Plot(TRAVELTIME.value, slope_break_method = 1, breakpoints = selected_breakpoints)
        except ValueError as e:
            print(f"Manual revision failed: {e}")

# Accept Function
def Accept (_):

    global VS_SLOPEBREAK_DF

    travelTimeMeta_ID = TRAVELTIME.value
    row = REVIEW_DF['travelTimeMeta_ID'] == travelTimeMeta_ID

    # Update the DataFrame
    REVIEW_DF.loc[row, 'status'] = 'accepted'
    REVIEW_DF.loc[row, 'VsZ_Direct_interpretation'] = VsZ_input.value
    # REVIEW_DF.loc[row, 'VsZp_Direct_interpretation'] = VsZp_input.value
    REVIEW_DF.loc[row, 'VsZ_Global'] = VsZ_model_input.value
    REVIEW_DF.loc[row, 'VsZ_Site_Specific'] = VsZ_ss_input.value
    REVIEW_DF.loc[row, 'VsZ_Robertson'] = VsZ_rob_input.value
    REVIEW_DF.loc[row, 'comment'] = COMMENT.value

    # Save to CSV
    REVIEW_DF.to_csv('traveltime_review.csv', index=False)

    # Save the LBR Vs profile to a CSV file
    new_data2 = pd.DataFrame({'travelTimeMeta_ID': np.full(len(velocity_updated), travelTimeMeta_ID),
                              'Velocity': velocity_updated,
                              'Top_depth': top_depth_updated,
                              'Bottom_depth': bottom_depth_updated})
    VS_SLOPEBREAK_DF = pd.concat([VS_SLOPEBREAK_DF, new_data2], ignore_index=True)
    VS_SLOPEBREAK_DF.to_csv('slopebreak_vs.csv', index=False)

    # Move to the next ID
    try:
        TRAVELTIME.value = REVIEW_DF[REVIEW_DF['status'] == 'Pending']['travelTimeMeta_ID'].values[0]
    except IndexError:
        print("No more IDs to process.")

def Reject (_):
    travelTimeMeta_ID = TRAVELTIME.value
    # Update the status and comment: invalid CPT or Traveltime data
    REVIEW_DF.loc[REVIEW_DF['travelTimeMeta_ID'] == travelTimeMeta_ID, ['status', 'comment']] = ['rejected', COMMENT.value]
    REVIEW_DF.to_csv('traveltime_review.csv', index=False)

# Save the current profile figure at 300 DPI, named by traveltime ID
def Save_Figure (_):
    if current_fig is None:
        with out:
            print("No figure to save.")
        return
    filename = f'profile_tt_{TRAVELTIME.value}.png'
    current_fig.savefig(filename, dpi=300, bbox_inches='tight')
    with out:
        print(f"Figure saved as {filename}")


# Initial display
display(SEARCH)
display(out)

# Trigger first update manually
Update_SCPT(None)

# Attach trigger to dropdown
TRAVELTIME.observe(Update_SCPT, names=['value'])
ACCEPT.on_click(Accept)
REVISE_FULL_HAND.on_click(Full_Manual)
REJECT.on_click(Reject)
SAVE_FIG.on_click(Save_Figure)

c:\ProgramData\anaconda3\Lib\site-packages\jupyter_client\session.py:721: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


Output()